# Final-plan spatial distortion audit
Audit all unordered pairs of distinct positive-support edges, including shared-source/target pairs. This compares direct pairwise distances against a recorded cutoff; it does not test the different anchor-distance constraints used by recovery.


## Data and local paths
Data are not uploaded to GitHub. Download sources and preparation instructions: [DATA.md](DATA.md).
Default prepared-input folder: `data/revision/dlpfc_0708/` in the repository; set `RAFTUP_TUTORIAL_DATA` to override it. Relative paths inside `pair.json` resolve against that file. Outputs go to `outputs/revision/07_spatial_distortion/`; set `RAFTUP_OUTPUT_ROOT` to override. The real-data notebooks do not substitute synthetic data when inputs are missing.


In [ ]:
from pathlib import Path
import os, json
import numpy as np
import raftup
from raftup import run_automated_alignment, run_manual_alignment
from raftup.experiments import load_pair, save_alignment, coverage
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), Path.cwd())
DATA = Path(os.environ.get('RAFTUP_TUTORIAL_DATA', str(REPO / 'data/revision/dlpfc_0708'))).expanduser().resolve()
OUT = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', str(REPO / 'outputs/revision/07_spatial_distortion'))).expanduser().resolve()
print('Inputs:', DATA, '\nOutputs:', OUT)
print('RAFT-UP:', raftup.__version__, raftup.__file__)


In [ ]:
# Set DATA or RAFTUP_TUTORIAL_DATA to your prepared input directory.
# pair.json contains file paths, exact cost-matrix cell order and spatial calibration.
a, b, cost, cfg = load_pair(DATA / 'pair.json')
print(a.shape, b.shape, cost.shape)


In [ ]:
from scipy import sparse
from raftup.experiments import spatial_distortion
P = sparse.load_npz(DATA / 'alignment_matrix.npz')
# These must be recorded in pair.json, not inferred from a fitted transform.
# Scale original coordinates into the same working units as the saved cutoff.
scale = np.asarray(cfg['audit_coordinate_scale'], float)  # [A multiplier, B multiplier]
report = spatial_distortion(P, a.obsm['spatial'] * scale[0],
    b.obsm['spatial'] * scale[1], cutoff=cfg['audit_spatial_cutoff'], block_size=256)
print(report)


This is an exact quadratic audit; large plans can take substantial time. For the 51 archived experiments and equal-weight dataset summary, use `figures/05_spatial_distortion.ipynb`. Excess bins use the supplied coordinate units (µm where calibrated).
